# Worked solutions: 01-dvc exercises

This notebook answers the five exercises at the end of the
[01-dvc README](README.md) by running each one for real.

**How to use it:** run it top to bottom (about 2 minutes). It builds a throwaway
project in a temporary folder, with its own git repo, DVC setup and remote,
and recreates the state you have after Parts 1-3 of the lesson (12 months of
data, tags `v1` and `v2-data`). Each exercise starts from a fresh copy of that state.
It works in a temporary folder and changes nothing of yours: your lesson files,
your git repo and your `.dvc` folder are not touched.

## Setup: build the throwaway project

In [1]:
import os, sys, shutil, subprocess, tempfile, json, hashlib
from pathlib import Path

LESSON = Path.cwd().resolve()                 # the real lesson folder (we only read from it)
SIM = LESSON.parent / "00-data" / "simulate_emr.py"
ROOT = Path(tempfile.mkdtemp(prefix="dvc_exercises_"))   # everything happens in here

# Make sure `python` and `dvc` used by the pipeline come from this environment.
ENV = {**os.environ,
       "PATH": str(Path(sys.executable).parent) + os.pathsep + os.environ["PATH"],
       "DVC_NO_ANALYTICS": "1", "NO_COLOR": "1", "PYTHONWARNINGS": "ignore"}

def sh(cmd, cwd, keep=None, check=True):
    """Run a shell command, print it and its output. `keep` filters output lines."""
    r = subprocess.run(cmd, shell=True, cwd=cwd, env=ENV, text=True,
                       stdout=subprocess.PIPE, stderr=subprocess.STDOUT)
    print(f"$ {cmd}")
    for line in r.stdout.splitlines():
        if line.strip() and "Checking graph" not in line and "[?25" not in line \
                and (keep is None or keep(line)):
            print("  " + line)
    if check and r.returncode:
        raise RuntimeError(f"command failed ({r.returncode}): {cmd}")
    return r

# --- build the base project: Parts 1-3 of the README ---
BASE = ROOT / "base"
P = BASE / "project"
P.mkdir(parents=True)
for f in ["dvc.yaml", "params.yaml"]:
    shutil.copy(LESSON / f, P / f)
shutil.copytree(LESSON / "src", P / "src", ignore=shutil.ignore_patterns("__pycache__"))
(P / ".gitignore").write_text("__pycache__/\n")

quiet = lambda line: False                    # hide the output of setup steps
sh("git init -q -b main && git config user.name Workshop && git config user.email w@example.org", P, quiet)
sh("dvc init -q", P, quiet)
sh("git add -A && git commit -q -m 'Initialise DVC'", P, quiet)

# Part 1: first six months, dvc add, remote, push
sh(f"python {SIM} --start 2024-01 --months 6 --out data/raw", P, quiet)
sh("dvc add data/raw", P, quiet)
sh("git add data/raw.dvc data/.gitignore && git commit -q -m 'EMR visits Jan-Jun 2024'", P, quiet)
sh("dvc remote add -d storage ../remote", P, quiet)
sh("git add .dvc/config && git commit -q -m 'Add remote'", P, quiet)
sh("dvc push", P, quiet)

# Part 2: pipeline, commit, tag v1
sh("dvc repro", P, quiet)
sh("git add -A && git commit -q -m 'Baseline model' && git tag v1", P, quiet)
sh("dvc push", P, quiet)

# Part 3: Jul-Dec arrive
sh(f"python {SIM} --start 2024-07 --months 6 --out data/raw", P, quiet)
sh("dvc add data/raw && dvc push && dvc repro", P, quiet)
sh("git add -A && git commit -q -m 'EMR visits Jul-Dec 2024' && git tag v2-data", P, quiet)
sh("dvc push", P, quiet)

print("Base project ready in", BASE)
sh("git log --oneline --decorate", P)
sh("dvc status", P)
print("raw files:", len(list((P / "data/raw").glob("*.csv"))))
print("baseline metrics:", json.dumps(json.load(open(P / "metrics.json"))))

$ git init -q -b main && git config user.name Workshop && git config user.email w@example.org


$ dvc init -q
$ git add -A && git commit -q -m 'Initialise DVC'


$ python /home/user/mlops-workshop/malnutrition/00-data/simulate_emr.py --start 2024-01 --months 6 --out data/raw


$ dvc add data/raw
$ git add data/raw.dvc data/.gitignore && git commit -q -m 'EMR visits Jan-Jun 2024'


$ dvc remote add -d storage ../remote
$ git add .dvc/config && git commit -q -m 'Add remote'


...

$ dvc push


$ python /home/user/mlops-workshop/malnutrition/00-data/simulate_emr.py --start 2024-07 --months 6 --out data/raw


$ dvc add data/raw && dvc push && dvc repro
$ git add -A && git commit -q -m 'EMR visits Jul-Dec 2024' && git tag v2-data


$ dvc push
Base project ready in /tmp/dvc_exercises_ql6siths/base
$ git log --oneline --decorate
  edf68a7 (HEAD -> main, tag: v2-data) EMR visits Jul-Dec 2024
  d0e7408 (tag: v1) Baseline model
  c44fea2 Add remote
  b55ce1c EMR visits Jan-Jun 2024
  95c0b29 Initialise DVC


$ dvc status
  Data and pipelines are up to date.
raw files: 12
baseline metrics: {"n": 4000, "prevalence": 0.0907, "roc_auc": 0.8122, "sensitivity": 0.5702, "ppv": 0.2587, "alert_rate": 0.2, "pr_auc": 0.3375}


In [2]:
def fresh(n):
    """Give exercise n its own copy of the base project (and its remote)."""
    dest = ROOT / f"ex{n}"
    shutil.copytree(BASE, dest, symlinks=True)
    print(f"Exercise {n} starts from a fresh copy of the lesson state.")
    return dest / "project"

def stage_log(r):
    """Which stages ran, and which were skipped, from `dvc repro` output."""
    ran = [l.split("'")[1] for l in r.stdout.splitlines() if l.startswith("Running stage")]
    skipped = [l.split("'")[1] for l in r.stdout.splitlines()
               if "didn't change, skipping" in l and l.startswith("Stage")]
    print(f"\n>>> RAN: {ran}    SKIPPED: {skipped}")

show_stage = lambda l: ("Running stage" in l or "skipping" in l
                        or l.startswith(("train:", "trained", "overall", "OVERALL")))

## Exercise 1

Set `prepare.test_months: 3`. Which stages rerun? Why does `train` rerun?

In [3]:
P = fresh(1)
sh("wc -l data/processed/train.csv data/processed/test.csv", P)

# change the parameter (same as editing params.yaml by hand)
txt = (P / "params.yaml").read_text().replace("test_months: 2", "test_months: 3")
(P / "params.yaml").write_text(txt)

sh("dvc params diff", P)
sh("dvc status", P)                             # what DVC thinks is out of date, before running
r = sh("dvc repro", P, keep=show_stage)
stage_log(r)
sh("wc -l data/processed/train.csv data/processed/test.csv", P)

Exercise 1 starts from a fresh copy of the lesson state.
$ wc -l data/processed/train.csv data/processed/test.csv
    20001 data/processed/train.csv
     4001 data/processed/test.csv
    24002 total


$ dvc params diff
  Path         Param                HEAD    workspace
  params.yaml  prepare.test_months  2       3


$ dvc status
  prepare:
  	changed deps:
  		params.yaml:
  			modified:           prepare


$ dvc repro
  'data/raw.dvc' didn't change, skipping
  Running stage 'prepare':
  train: 2024-01..2024-09 (18000 visits)  test: 2024-10..2024-12 (6000 visits)
  Running stage 'train':
  trained logistic on 18000 visits
  Running stage 'evaluate':
  overall: {"n": 6000, "prevalence": 0.0885, "roc_auc": 0.7962, "sensitivity": 0.565, "ppv": 0.25, "alert_rate": 0.2, "pr_auc": 0.2986}

>>> RAN: ['prepare', 'train', 'evaluate']    SKIPPED: []
$ wc -l data/processed/train.csv data/processed/test.csv
    18001 data/processed/train.csv
     6001 data/processed/test.csv
    24002 total


CompletedProcess(args='wc -l data/processed/train.csv data/processed/test.csv', returncode=0, stdout='  18001 data/processed/train.csv\n   6001 data/processed/test.csv\n  24002 total\n')

**Answer**

`prepare`, `train` and `evaluate` all rerun (the log above says
`RAN: ['prepare', 'train', 'evaluate']`).

- `prepare` reruns because `params.yaml` changed in the `prepare` section it declares.
- `train` reruns even though its own parameters did not change. Its input
  `data/processed/train.csv` did: with 3 test months, training shrank from
  20,000 to 18,000 visits (Jan-Sep instead of Jan-Oct), so the file's hash changed.
- `evaluate` reruns because `test.csv` grew from 4,000 to 6,000 visits and the model changed.

DVC follows file hashes: a stage reruns when any of its inputs (code, data, params)
has a new hash, and the stages that read its outputs follow.

## Exercise 2

Edit only `src/evaluate.py` (e.g. change a print). Which stages rerun?

In [4]:
P = fresh(2)
# change one print message in evaluate.py
src = (P / "src/evaluate.py").read_text()
(P / "src/evaluate.py").write_text(src.replace('print("overall:"', 'print("OVERALL RESULTS:"'))
sh("git diff --stat", P)
sh("dvc status", P)                              # only evaluate is out of date
r = sh("dvc repro", P, keep=show_stage)
stage_log(r)
sh("dvc metrics diff --targets metrics.json", P)  # empty: the numbers did not change

Exercise 2 starts from a fresh copy of the lesson state.
$ git diff --stat
   src/evaluate.py | 2 +-
   1 file changed, 1 insertion(+), 1 deletion(-)


$ dvc status
  evaluate:
  	changed deps:
  		modified:           src/evaluate.py


$ dvc repro
  'data/raw.dvc' didn't change, skipping
  Stage 'prepare' didn't change, skipping
  Stage 'train' didn't change, skipping
  Running stage 'evaluate':
  OVERALL RESULTS: {"n": 4000, "prevalence": 0.0907, "roc_auc": 0.8122, "sensitivity": 0.5702, "ppv": 0.2587, "alert_rate": 0.2, "pr_auc": 0.3375}

>>> RAN: ['evaluate']    SKIPPED: ['prepare', 'train']


$ dvc metrics diff --targets metrics.json


CompletedProcess(args='dvc metrics diff --targets metrics.json', returncode=0, stdout='')

**Answer**

Only `evaluate` reruns. `prepare` and `train` are skipped, because neither of them
reads `src/evaluate.py`.

`dvc status` showed this before running: only `evaluate` was out of date. DVC hashes the
file's content, so any edit counts, even a print or a comment. The metrics did not change
(`dvc metrics diff` printed nothing): DVC cannot know the edit was harmless, so it reruns to be safe.

## Exercise 3

Change `evaluate.alert_rate` to 0.3. What happens to sensitivity and PPV,
and what does that mean for a clinic's workload?

In [5]:
P = fresh(3)
txt = (P / "params.yaml").read_text().replace("alert_rate: 0.2 ", "alert_rate: 0.3 ")
(P / "params.yaml").write_text(txt)
r = sh("dvc repro", P, keep=show_stage)
stage_log(r)
sh("dvc metrics diff --targets metrics.json", P)

# what the numbers mean in children, for the 4000 test visits
new = json.load(open(P / "metrics.json"))
old = json.loads(subprocess.check_output("git show HEAD:metrics.json", shell=True, cwd=P, text=True))
for name, x in [("alert_rate 0.2", old), ("alert_rate 0.3", new)]:
    flagged = round(x["n"] * x["alert_rate"])
    print(f"{name}: {flagged} children flagged, {round(flagged * x['ppv'])} of them true GAM cases, "
          f"{round(flagged * (1 - x['ppv']))} false alarms")

Exercise 3 starts from a fresh copy of the lesson state.


$ dvc repro
  'data/raw.dvc' didn't change, skipping
  Stage 'prepare' didn't change, skipping
  Stage 'train' didn't change, skipping
  Running stage 'evaluate':
  overall: {"n": 4000, "prevalence": 0.0907, "roc_auc": 0.8122, "sensitivity": 0.7135, "ppv": 0.2158, "alert_rate": 0.3, "pr_auc": 0.3375}

>>> RAN: ['evaluate']    SKIPPED: ['prepare', 'train']


$ dvc metrics diff --targets metrics.json
  Path          Metric       HEAD    workspace    Change
  metrics.json  alert_rate   0.2     0.3          0.1
  metrics.json  ppv          0.2587  0.2158       -0.0429
  metrics.json  sensitivity  0.5702  0.7135       0.1433
alert_rate 0.2: 800 children flagged, 207 of them true GAM cases, 593 false alarms
alert_rate 0.3: 1200 children flagged, 259 of them true GAM cases, 941 false alarms


**Answer**

Only `evaluate` reruns (`alert_rate` is only read there). The model is the same, but the
cut-off moves so that the top 30% of visits are flagged instead of the top 20%.

- Sensitivity goes up, from 0.570 to 0.714: we catch more of the children who develop GAM.
- PPV goes down, from 0.259 to 0.216: a smaller share of flagged children are true cases.

For the clinic: of the 4,000 test visits, 1,200 children must be followed up instead of 800 (50% more work).
That finds about 52 more true cases (259 vs 207), but about 348 more
false alarms. The model is not better, we just chose to spend more capacity. The right
`alert_rate` is a decision about how many follow-ups the clinic can actually do.

## Exercise 4

Add month 2025-01 with `--scenario new_regions`. How many files does
`dvc push` upload, and why?

In [6]:
P = fresh(4)
sh(f"python {SIM} --start 2025-01 --scenario new_regions --out data/raw", P)
sh("dvc status", P)
sh("dvc add data/raw", P)
count = lambda: sum(1 for x in (P.parent / "remote").rglob("*") if x.is_file())
print("files in remote before:", count())
sh("dvc push", P)
print("files in remote after: ", count())

# Extra: if we also rerun the pipeline, push has more to upload (processed data + model).
sh("git add -A && git commit -q -m 'add 2025-01'", P, quiet)
sh("dvc repro", P, lambda l: "Running stage" in l)
sh("dvc push", P)

Exercise 4 starts from a fresh copy of the lesson state.


$ python /home/user/mlops-workshop/malnutrition/00-data/simulate_emr.py --start 2025-01 --scenario new_regions --out data/raw
  data/raw/emr_2025-01.csv: 2000 visits, 11.8% GAM within 90d [new_regions]


$ dvc status
  prepare:
  	changed deps:
  		modified:           data/raw
  data/raw.dvc:
  	changed outs:
  		modified:           data/raw


$ dvc add data/raw
  To track the changes with git, run:
  	git add data/raw.dvc
  To enable auto staging, run:
  	dvc config core.autostage true
files in remote before: 22


$ dvc push
  2 files pushed
files in remote after:  24
$ git add -A && git commit -q -m 'add 2025-01'


$ dvc repro
  Running stage 'prepare':
  Running stage 'train':
  Running stage 'evaluate':


$ dvc push
  4 files pushed


CompletedProcess(args='dvc push', returncode=0, stdout='4 files pushed\n')

**Answer**

`dvc push` uploads **2 files**: the new `emr_2025-01.csv` and the new directory manifest
(the `.dir` file that lists the folder's contents).

DVC stores files by content hash, and the remote already holds the 12 earlier CSVs, so
they are not sent again. Adding a month changes the folder's hash, which needs a new
manifest. The remote's file count above went up by exactly 2.

The push happened before `dvc repro`, so the processed data and model were not part of it.
After the extra `dvc repro`, the second push uploads those new outputs.

## Exercise 5

Delete `data/raw/emr_2024-03.csv` by accident. How do you get it back?
(Hint: `dvc checkout`.) What if you also deleted `.dvc/cache`?

In [7]:
P = fresh(5)
f = P / "data/raw/emr_2024-03.csv"
md5 = lambda p: hashlib.md5(p.read_bytes()).hexdigest()
before = md5(f)

f.unlink()                                       # the accident
sh("ls data/raw | wc -l", P)
sh("dvc status", P)                              # DVC notices the missing file

Exercise 5 starts from a fresh copy of the lesson state.
$ ls data/raw | wc -l
  11


$ dvc status
  prepare:
  	changed deps:
  		modified:           data/raw
  data/raw.dvc:
  	changed outs:
  		modified:           data/raw


CompletedProcess(args='dvc status', returncode=0, stdout='prepare:\n\tchanged deps:\n\t\tmodified:           data/raw\ndata/raw.dvc:\n\tchanged outs:\n\t\tmodified:           data/raw\n')

In [8]:
# Fix 1: the file is still in .dvc/cache, so dvc checkout copies it back.
sh("dvc checkout", P)
print("restored:", f.exists(), "| identical to before:", md5(f) == before)
sh("git status --short", P)                      # git sees nothing changed

$ dvc checkout
  M       data/raw/
restored: True | identical to before: True
$ git status --short


CompletedProcess(args='git status --short', returncode=0, stdout='')

In [9]:
# Worse: delete the file AND the local cache.
f.unlink()
shutil.rmtree(P / ".dvc/cache")
r = sh("dvc checkout", P, check=False)           # the bytes are nowhere on this machine
print("file back?", f.exists(), "| csv files left in data/raw:", len(list((P / "data/raw").glob("*.csv"))))

$ dvc checkout
  ERROR: Checkout failed for following targets:
  data/raw
  Is your cache up to date?
  <https://error.dvc.org/missing-files>
file back? False | csv files left in data/raw: 0


In [10]:
# Fix 2: the remote still has everything (we ran dvc push), so dvc pull fetches it.
sh("dvc pull", P)
print("restored:", f.exists(), "| identical to before:", md5(f) == before)
sh("dvc status", P)

$ dvc pull
  A       data/raw/
  17 files fetched and 12 files added
restored: True | identical to before: True


$ dvc status
  Data and pipelines are up to date.


CompletedProcess(args='dvc status', returncode=0, stdout='Data and pipelines are up to date.\n')

**Answer**

1. **Cache still there:** run `dvc checkout`. DVC sees that `data/raw` no longer matches
   `data/raw.dvc` and copies the file back from `.dvc/cache`. The checksum is identical to the
   original, and git shows no change, since the pointer file never changed.
2. **Cache deleted as well:** `dvc checkout` fails with "Checkout failed ... Is your cache up to date?"
   and the `data/raw` folder stays empty, because the bytes are no longer on this machine.
   Run `dvc pull`: it fetches them from the remote, rebuilds the cache, and restores the file.
   This only works because the data was pushed earlier with `dvc push`. Without a push, the file would be gone for good.

Done. The temporary folder is removed below.

In [11]:
shutil.rmtree(ROOT, ignore_errors=True)
print('removed', ROOT)

removed /tmp/dvc_exercises_ql6siths
